# CSIS3764 Exam Q3 — End-of-Year 2022
## Wine Dataset — Unsupervised Learning

## 3.1 — Import dataset

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

wine = pd.read_csv('wine.csv')
print(wine.head())

## 3.2 — Inspect the data

In [ ]:
# First 20 records
print(wine.head(20))

# Concise summary
wine.info()

# Check for zero values
print('Zero values per column:')
print((wine == 0).sum())

## 3.3 — k-Means clustering on selected features

In [ ]:
# Select only required features
features = ['Alcohol', 'Magnesium', 'Total_Phenols', 'Flavanoids', 'Color_Intensity']
wine_features = wine[features].copy()
print(wine_features.head())

# Scale features
scaler = StandardScaler()
wine_scaled = scaler.fit_transform(wine_features)
print('Scaled shape:', wine_scaled.shape)

In [ ]:
# Elbow curve + silhouette
k_range = range(2, 11)
inertia = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(wine_scaled)
    inertia.append(km.inertia_)
    sil_scores.append(silhouette_score(wine_scaled, labels))

plt.figure(figsize=(8, 5))
plt.plot(k_range, inertia, marker='o', color='blue')
plt.title('Elbow Curve')
plt.xlabel('k')
plt.ylabel('Inertia')
plt.xticks(k_range)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(k_range, sil_scores, marker='o', color='orange')
plt.title('Silhouette Score per k')
plt.xlabel('k')
plt.ylabel('Silhouette Score')
plt.xticks(k_range)
plt.tight_layout()
plt.show()

best_k = list(k_range)[sil_scores.index(max(sil_scores))]
print(f'Best k: {best_k} (silhouette = {max(sil_scores):.4f})')

In [ ]:
# Train final model
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
wine_features['Cluster'] = kmeans.fit_predict(wine_scaled)

print('Cluster labels:')
print(wine_features['Cluster'].values)

# Cluster centres (inverse transform to original scale)
centres = scaler.inverse_transform(kmeans.cluster_centers_)
centres_df = pd.DataFrame(centres, columns=features)
centres_df.index.name = 'Cluster'
print('\nCluster centres:')
print(centres_df)

## 3.4 — Scatter plots with clusters and centroids

In [ ]:
pairs = [(features[i], features[j])
         for i in range(len(features))
         for j in range(i+1, len(features))]

palette = sns.color_palette('tab10', n_colors=best_k)

for x_feat, y_feat in pairs:
    plt.figure(figsize=(7, 5))
    sns.scatterplot(
        data=wine_features, x=x_feat, y=y_feat,
        hue='Cluster', palette=palette, alpha=0.7)
    for i, row in centres_df.iterrows():
        plt.scatter(row[x_feat], row[y_feat],
                    color=palette[i], marker='X', s=200,
                    edgecolors='black', linewidths=1.5)
    plt.title(f'{x_feat} vs {y_feat}')
    plt.tight_layout()
    plt.show()

## 3.5 — Evaluation and discussion

In [ ]:
print(f"""
=== Cluster Evaluation ===

Optimal k from silhouette score: {best_k}

Elbow Curve:
The elbow curve shows where inertia stops dropping significantly.
The bend in the curve suggests the optimal k where adding more clusters
yields diminishing returns.

Silhouette Score:
The highest silhouette score at k={best_k} indicates that data points
are well matched to their own cluster and clearly separated from
neighbouring clusters at this value of k.

Scatter Plots:
If the scatter plots show {best_k} visually distinct and well-separated
groups of points, this confirms that k={best_k} is appropriate.
If clusters overlap in some scatter plots but are well separated in
others, it suggests those particular feature pairs are more informative
for distinguishing wine types than others.
""")